In [2]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels

import pandas as pd
import numpy as np
import pyblp 
import statsmodels.formula.api as smf

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


'1.1.2'

In [3]:
################ Load data ready for nested logit
df = pd.read_csv('data_with_IV.csv')

# Generate nesting_id
df['nesting_ids'] = pd.factorize(df['fuel_type'])[0]

# Generate log of charging station stock
df['log_charging_IV'] = np.log(df['charging_stations_stock_lag']) 

# Create indicator for electric vehicles
df['is_electric'] = df['type'].apply(lambda x: 1 if x == '国产新能源乘用车' else 0)

# Make year an object variable
df['year'] = df['year'].astype(str)


In [4]:
# Rename columns to match pyblp requirements
df.rename(columns={
    'product_id': 'product_ids',
    'market_id': 'market_ids',
    'weighted_Avg_Price': 'prices',
    'market_share': 'shares',
    'cost_shifter' : 'demand_instruments0',
}, inplace=True)


In [5]:
################ Nested Logit Model with Charging
def solve_nl(df):
    groups = df.groupby(['market_ids', 'nesting_ids'])
    df['demand_instruments1'] = groups['shares'].transform(np.size)
    nl_formulation = pyblp.Formulation('0 + prices + is_electric*log_charging_IV')
    problem = pyblp.Problem(nl_formulation, df)
    return problem.solve(rho=0.66)

# Solve the nested logit model
results = solve_nl(df)

# Display results
print(results)

Problem Results Summary:
GMM   Objective    Projected    Reduced   Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Hessian   Shares   Condition Number  Condition Number 
----  ---------  -------------  --------  -------  ----------------  -----------------
 2    +9.2E-22     +8.3E-09     +2.7E+04     0         +1.8E+13          +1.3E+05     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective 
   Time      Converged   Iterations   Evaluations
-----------  ---------  ------------  -----------
 00:01:44       Yes          4            13     

Rho Estimates (Robust SEs in Parentheses):
All Groups
----------
 +6.6E-01 
(+8.6E-03)

Beta Estimates (Robust SEs in Parentheses):
  prices    is_electric  log_charging_IV  is_electric*log_charging_IV
----------  -----------  ---------------  ---------------------------
 -8.1E-02    -1.2E+01       -5.4E-01               +1.1E+00          
(+1.2E-03)  (+1.0E-01)     (+6.1E-03)             (+1.1E-02)

In [ ]:
################ Compute for own- and cross-price elasticities
market = df['market_ids' == 'P01Y2019']
elasticities = results.compute_elasticities()
print(elasticities[market])